In [42]:
!pip install bs4


   ---------------------------------------- 0/3 [soupsieve]
   ------------- -------------------------- 1/3 [beautifulsoup4]
   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]




[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
# imports 
import ast
import difflib
import hashlib
import json
import math
import os
import re
import unicodedata
from pathlib import Path
from typing import Annotated, Any
from urllib.parse import quote, quote_plus, urljoin, urlparse
from difflib import SequenceMatcher
from functools import lru_cache

import numexpr
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

import ifcopenshell

In [45]:
def load_llm(id_model, temperature):
    llm = ChatOpenAI(
        model=id_model,
        temperature=temperature,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    return llm

In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("openai_api_key")
if not openai_api_key:
    raise EnvironmentError("Defina OPENAI_API_KEY ou openai_api_key antes de criar o LLM.")
os.environ["OPENAI_API_KEY"] = openai_api_key
id_model = "gpt-4.1"
temperature = 0.2

llm = load_llm(id_model, temperature)

In [95]:
# Math tools
@tool
def calculator_tool(expression: str) -> str:
    """Use para qualquer calculo aritmetico direto; não faça cálculo numérico de cabeça."""
    local_dict = {"pi": math.pi, "e": math.e}
    return str(
        numexpr.evaluate(
            expression.strip(),
            global_dict={},
            local_dict=local_dict,
        )
    )


@tool
def percentage_tool(base_value: float, percent: float) -> dict:
    """Use para calcular porcentagem aplicada sobre um valor base; não faça cálculo de cabeça."""
    percentage_value = base_value * (percent / 100)
    total_with_percentage = base_value + percentage_value
    return {
        "base_value": base_value,
        "percent": percent,
        "percentage_value": percentage_value,
        "total_with_percentage": total_with_percentage,
    }


@tool
def percent_change_tool(initial_value: float, final_value: float) -> dict:
    """Use para calcular variação percentual entre dois valores; não faça cálculo de cabeça."""
    if initial_value == 0:
        return {
            "error": "initial_value não pode ser zero para calcular variação percentual."
        }

    absolute_change = final_value - initial_value
    percent_change = (absolute_change / initial_value) * 100
    return {
        "initial_value": initial_value,
        "final_value": final_value,
        "absolute_change": absolute_change,
        "percent_change": percent_change,
    }


def _parse_amount_unit(unit_text: str | None) -> tuple[float | None, str | None]:
    if not unit_text:
        return None, None

    text = unit_text.casefold().replace(",", ".")
    patterns = [
        (r"(\d+(?:\.\d+)?)\s*(?:m²|m2|metros?\s+quadrados?)\b", "m2", 1),
        (r"(\d+(?:\.\d+)?)\s*(?:ml|mililitros?)\b", "L", 0.001),
        (r"(\d+(?:\.\d+)?)\s*(?:litros?|lts?|lt|l)\b", "L", 1),
        (r"(\d+(?:\.\d+)?)\s*(?:gramas?|g)\b", "kg", 0.001),
        (r"(\d+(?:\.\d+)?)\s*(?:quilogramas?|kilos?|kg)\b", "kg", 1),
        (r"(\d+(?:\.\d+)?)\s*(?:metros?|m)\b", "m", 1),
        (r"(\d+(?:\.\d+)?)\s*(?:unidades?|und|un)\b", "un", 1),
    ]
    for pattern, family, multiplier in patterns:
        match = re.search(pattern, text)
        if match:
            return float(match.group(1)) * multiplier, family
    if re.search(r"\b(?:unidades?|und|un)\b", text):
        return 1, "un"
    return None, None


def _parse_unit_family(unit_text: str | None) -> str | None:
    _amount, family = _parse_amount_unit(f"1 {unit_text}" if unit_text else None)
    return family


@tool
def purchase_quantity_tool(
    required_quantity: float,
    required_unit: str,
    offer_unit: str,
    unit_price: float | None = None,
) -> dict:
    """Calculate how many commercial packages to buy based on required amount and offer unit."""
    required_family = _parse_unit_family(required_unit)
    offer_size, offer_family = _parse_amount_unit(offer_unit)

    if required_quantity is None or required_quantity <= 0:
        return {"error": "required_quantity must be greater than zero."}
    if not required_family:
        return {"error": f"Could not infer required unit family from '{required_unit}'."}
    if offer_size is None or not offer_family:
        return {"error": f"Could not infer package size from offer_unit '{offer_unit}'."}
    if required_family != offer_family:
        return {
            "error": "required_unit and offer_unit are not compatible.",
            "required_unit": required_unit,
            "offer_unit": offer_unit,
            "required_family": required_family,
            "offer_family": offer_family,
        }

    purchase_quantity = math.ceil(required_quantity / offer_size)
    covered_quantity = purchase_quantity * offer_size
    total_price = unit_price * purchase_quantity if unit_price is not None else None
    return {
        "required_quantity": required_quantity,
        "required_unit": required_unit,
        "offer_unit": offer_unit,
        "package_size": offer_size,
        "unit_family": required_family,
        "purchase_quantity": purchase_quantity,
        "covered_quantity": covered_quantity,
        "unit_price": unit_price,
        "total_price": total_price,
    }

In [96]:
# Supplier models and placeholder search tool
import sys
from datetime import date
from typing import Any

from langchain_core.messages import HumanMessage, SystemMessage

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "backend/api/src/app/models/materials.py").exists():
    project_root = project_root.parent

api_src_path = project_root / "backend/api/src"
if str(api_src_path) not in sys.path:
    sys.path.insert(0, str(api_src_path))

from app.models.materials import ListaMateriaisObra, MaterialObra, OfertaFornecedor


def _infer_commercial_unit(text: str | None, fallback_unit: str = "") -> str | None:
    """Infer the commercial package unit from a shopping result title."""
    if not text:
        return fallback_unit or None

    unit_patterns = [
        r"(\d+(?:[\.,]\d+)?)\s*(?:litros?|lts?|lt|l)\b",
        r"(\d+(?:[\.,]\d+)?)\s*(?:quilogramas?|kilos?|kg)\b",
        r"(\d+(?:[\.,]\d+)?)\s*(?:gramas?|g)\b",
        r"(\d+(?:[\.,]\d+)?)\s*(?:metros?\s*quadrados?|m²|m2)\b",
        r"(\d+(?:[\.,]\d+)?)\s*(?:metros?|m)\b",
        r"(\d+(?:[\.,]\d+)?)\s*(?:unidades?|und|un)\b",
    ]
    normalized_suffixes = ["L", "kg", "g", "m2", "m", "un"]

    for pattern, suffix in zip(unit_patterns, normalized_suffixes):
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            amount = match.group(1).replace(",", ".")
            return f"{amount} {suffix}"

    return fallback_unit or None


DEFAULT_SCRAPING_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
}


def _scraper_log(event: str, payload: Any | None = None) -> None:
    """Print compact scraper logs for debugging supplier search."""
    print(f"[scraper] {event}")
    if payload is None:
        return
    try:
        text = json.dumps(payload, ensure_ascii=False, default=str)
    except TypeError:
        text = str(payload)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) > 1200:
        text = text[:1200] + "... [truncated]"
    print(text)


def fetch_site_html(url: str, timeout: int = 20) -> str:
    """Fetch a site page and return its HTML."""
    _scraper_log("http_html_request", {"url": url, "timeout": timeout})
    response = requests.get(url, headers=DEFAULT_SCRAPING_HEADERS, timeout=timeout)
    _scraper_log(
        "http_html_response",
        {
            "url": url,
            "status_code": response.status_code,
            "content_type": response.headers.get("content-type"),
            "bytes": len(response.content or b""),
        },
    )
    response.raise_for_status()
    return response.text


def parse_html(html: str) -> BeautifulSoup:
    """Parse raw HTML into a BeautifulSoup document."""
    return BeautifulSoup(html, "html.parser")


def clean_scraped_text(value: Any) -> str:
    """Normalize whitespace from scraped text."""
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_search_text(value: str) -> str:
    """Normalize text for accent-insensitive matching."""
    normalized = unicodedata.normalize("NFKD", value or "")
    without_accents = "".join(char for char in normalized if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", " ", without_accents.casefold()).strip()


def text_match_score(query: str, *texts: str) -> float:
    """Score how well a group of scraped texts matches a query."""
    query_normalized = normalize_search_text(query)
    haystack = normalize_search_text(" ".join(texts))
    if not query_normalized:
        return 1.0
    if not haystack:
        return 0.0

    query_tokens = set(query_normalized.split())
    haystack_tokens = set(haystack.split())
    token_score = len(query_tokens & haystack_tokens) / max(len(query_tokens), 1)
    sequence_score = SequenceMatcher(None, query_normalized, haystack).ratio()
    return max(token_score, sequence_score)


def extract_links_by_path(soup: BeautifulSoup, base_url: str, path_fragment: str) -> list[dict]:
    """Extract unique links whose absolute URL contains path_fragment."""
    links_by_url: dict[str, dict] = {}
    for anchor in soup.find_all("a", href=True):
        absolute_url = urljoin(base_url, anchor.get("href"))
        if path_fragment not in absolute_url:
            continue

        text = clean_scraped_text(anchor.get_text(" "))
        item = links_by_url.setdefault(absolute_url, {"url": absolute_url, "text_candidates": []})
        if text and text.casefold() not in {"ver detalhes", "fazer orçamento"}:
            item["text_candidates"].append(text)

    return list(links_by_url.values())


def first_meaningful_heading(soup: BeautifulSoup, fallback: str = "") -> str:
    """Return the first heading that looks like actual page content."""
    ignored = {"produtos", "ficou com alguma dúvida?", "entre em contato?", "links", "contato"}
    for heading in soup.find_all(["h1", "h2", "h3"]):
        text = clean_scraped_text(heading.get_text(" "))
        if text and text.casefold() not in ignored:
            return text
    return fallback


def extract_meaningful_paragraphs(soup: BeautifulSoup, max_paragraphs: int = 2) -> str:
    """Extract a compact description from paragraphs/list items in a page."""
    ignored_fragments = ["©", "desenvolvido", "fale com", "contato@", "3217-7447"]
    paragraphs = []
    for node in soup.find_all(["p", "li"]):
        text = clean_scraped_text(node.get_text(" "))
        if len(text) < 35:
            continue
        if any(fragment in text.casefold() for fragment in ignored_fragments):
            continue
        paragraphs.append(text)
        if len(paragraphs) >= max_paragraphs:
            break
    return " ".join(paragraphs)


PISOLAR_BASE_URL = "https://www.pisolar.com.br/"


def _format_query_quantity(quantity: float | int | str | None) -> str:
    """Format quantity for storefront search terms."""
    if quantity is None or quantity == "":
        return ""
    try:
        numeric_quantity = float(quantity)
    except (TypeError, ValueError):
        return str(quantity).strip().replace(".", ",")
    if numeric_quantity.is_integer():
        return str(int(numeric_quantity))
    return str(numeric_quantity).rstrip("0").rstrip(".").replace(".", ",")


def _compact_unit_for_query(unit: str | None) -> str:
    """Convert verbose units into compact search tokens, e.g. litros -> L."""
    if not unit:
        return ""
    raw_unit = unit.strip().casefold().replace("²", "2")
    normalized_unit = normalize_search_text(raw_unit)
    compact_units = {
        "l": "L",
        "lt": "L",
        "lts": "L",
        "litro": "L",
        "litros": "L",
        "ml": "ml",
        "mililitro": "ml",
        "mililitros": "ml",
        "kg": "kg",
        "quilo": "kg",
        "quilos": "kg",
        "kilo": "kg",
        "kilos": "kg",
        "quilograma": "kg",
        "quilogramas": "kg",
        "g": "g",
        "grama": "g",
        "gramas": "g",
        "m2": "m2",
        "metro quadrado": "m2",
        "metros quadrados": "m2",
        "m": "m",
        "metro": "m",
        "metros": "m",
        "un": "un",
        "und": "un",
        "unidade": "un",
        "unidades": "un",
    }
    return compact_units.get(normalized_unit, unit.strip())


def compact_quantity_unit_for_query(quantity: float | int | str | None, unit: str = "") -> str:
    """Join quantity and unit in the storefront-friendly form, e.g. 30 + litros -> 30L."""
    quantity_text = _format_query_quantity(quantity)
    compact_unit = _compact_unit_for_query(unit)
    if quantity_text and compact_unit:
        return f"{quantity_text}{compact_unit}"
    if quantity_text:
        return quantity_text
    return compact_unit


def build_supplier_site_query(product_name: str, unit: str = "", quantity: float | None = None) -> str:
    """Build a store-search query like the search bar would receive, without city."""
    query_parts = [product_name]
    quantity_unit = compact_quantity_unit_for_query(quantity=quantity, unit=unit)
    if quantity_unit:
        query_parts.append(quantity_unit)
    return " ".join(str(part).strip() for part in query_parts if str(part).strip())


def build_supplier_search_queries(product_name: str, unit: str = "", quantity: float | None = None) -> list[str]:
    """Build increasingly broad storefront search queries, without city."""
    return list(
        dict.fromkeys(
            [
                build_supplier_site_query(product_name=product_name, unit=unit, quantity=quantity),
                build_supplier_site_query(product_name=product_name, unit="", quantity=quantity),
                build_supplier_site_query(product_name=product_name, unit=unit, quantity=None),
                product_name.strip(),
            ]
        )
    )


def _coerce_price(value: Any) -> float | None:
    """Coerce a scraped price-like value into float."""
    if value in (None, ""):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip()
    if not text:
        return None
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return parse_brazilian_price(text)


def _secure_same_site_url(base_url: str, raw_url: str | None) -> str | None:
    """Build an absolute same-site product URL and prefer https."""
    if not raw_url:
        return None
    absolute_url = urljoin(base_url, raw_url)
    if absolute_url.startswith("http://"):
        absolute_url = "https://" + absolute_url[len("http://") :]
    return absolute_url


def _extract_js_array_assignment(script_text: str, variable_name: str) -> str | None:
    """Extract a JavaScript array assigned to variable_name from a script string."""
    assignment = re.search(rf"\b{re.escape(variable_name)}\s*=", script_text)
    if not assignment:
        return None
    start = script_text.find("[", assignment.end())
    if start == -1:
        return None

    depth = 0
    in_string = False
    quote_char = ""
    escaped = False
    for position, char in enumerate(script_text[start:], start=start):
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == quote_char:
                in_string = False
            continue

        if char in {'"', "'"}:
            in_string = True
            quote_char = char
        elif char == "[":
            depth += 1
        elif char == "]":
            depth -= 1
            if depth == 0:
                return script_text[start : position + 1]
    return None


def extract_tray_datalayer_products(html: str) -> list[dict]:
    """Extract Tray dataLayer listProducts entries from a search/category page."""
    soup = parse_html(html)
    products: list[dict] = []
    for script in soup.find_all("script"):
        script_text = script.string or script.get_text("\n")
        if "dataLayer" not in script_text or "listProducts" not in script_text:
            continue
        array_text = _extract_js_array_assignment(script_text, "dataLayer")
        if not array_text:
            continue
        try:
            data_layer = json.loads(array_text)
        except json.JSONDecodeError as exc:
            _scraper_log("tray_datalayer_parse_failed", {"error": str(exc)})
            continue
        if not isinstance(data_layer, list):
            continue
        for item in data_layer:
            if isinstance(item, dict) and isinstance(item.get("listProducts"), list):
                products.extend(product for product in item["listProducts"] if isinstance(product, dict))
    return products


def tray_datalayer_product_to_offer(
    product: dict,
    supplier_name: str,
    base_url: str,
    fallback_unit: str = "",
    default_installments: int | None = None,
) -> dict | None:
    """Convert a Tray listProducts item into an OfertaFornecedor-shaped dict."""
    title = clean_scraped_text(product.get("nameProduct") or product.get("item_name") or "")
    if not title:
        return None
    link_produto = _secure_same_site_url(base_url, product.get("urlProduct") or product.get("item_url"))
    combined_text = " ".join(
        clean_scraped_text(value)
        for value in [title, product.get("category"), product.get("item_category"), product.get("item_category2")]
        if value
    )
    availability = product.get("availability")
    disponibilidade = "Disponivel" if availability in ("YES", "IN_STOCK", True) else "Indisponivel" if availability else None
    return {
        "fornecedor": supplier_name,
        "descricao": title,
        "marca": product.get("brand") or product.get("item_brand") or _infer_brand_from_title(title),
        "unidade": _infer_commercial_unit(combined_text, fallback_unit),
        "quantidade": None,
        "valor_unitario": _coerce_price(product.get("sellPrice") or product.get("price")),
        "valor_total": None,
        "preco_a_vista": None,
        "preco_a_prazo": None,
        "num_parcelas": default_installments,
        "frete": None,
        "disponibilidade": disponibilidade or "Produto encontrado na busca do fornecedor.",
        "data_consulta": date.today().isoformat(),
        "link_produto": link_produto,
    }


def scrape_tray_datalayer_offers(
    html: str,
    query: str,
    supplier_name: str,
    base_url: str,
    fallback_unit: str = "",
    default_installments: int | None = None,
    limit: int = 5,
) -> list[dict]:
    """Build supplier offers from Tray dataLayer products."""
    products = extract_tray_datalayer_products(html)
    _scraper_log(
        "tray_datalayer_products_extracted",
        {
            "supplier": supplier_name,
            "query": query,
            "products": len(products),
            "sample_titles": [product.get("nameProduct") or product.get("item_name") for product in products[:3]],
        },
    )
    offers = []
    for product in products:
        offer = tray_datalayer_product_to_offer(
            product=product,
            supplier_name=supplier_name,
            base_url=base_url,
            fallback_unit=fallback_unit,
            default_installments=default_installments,
        )
        if offer:
            offers.append(offer)
    return sorted(
        offers,
        key=lambda offer: text_match_score(query, offer.get("descricao", ""), offer.get("unidade", "")),
        reverse=True,
    )[:limit]


def scrape_schema_org_product_offers(
    html: str,
    query: str,
    supplier_name: str,
    base_url: str,
    fallback_unit: str = "",
    availability_text: str = "Produto encontrado na busca do fornecedor.",
    default_installments: int | None = None,
    limit: int = 5,
) -> list[dict]:
    """Build offers from schema.org Product cards in storefront HTML."""
    soup = parse_html(html)
    containers = soup.select(".product-box, [itemscope][itemtype*='Product']")
    offers_by_url: dict[str, dict] = {}

    for container in containers:
        title_node = (
            container.select_one('[itemprop="name"] strong')
            or container.select_one('[itemprop="name"]')
            or container.select_one('.product-name')
        )
        title = clean_scraped_text(title_node.get_text(" ") if title_node else "")
        if len(title) < 8:
            continue

        meta_url = container.select_one('meta[itemprop="url"][content]')
        link_node = (
            container.select_one('a[itemprop="url"][href]')
            or container.select_one('a.product-name[href]')
            or container.select_one('a[href]')
        )
        raw_url = meta_url.get("content") if meta_url else link_node.get("href") if link_node else None
        link_produto = _secure_same_site_url(base_url, raw_url)
        if not link_produto or not _same_site_url(link_produto, base_url):
            continue

        brand_node = (
            container.select_one('[itemprop="brand"] [itemprop="name"]')
            or container.select_one('[itemprop="brand"]')
        )
        brand = clean_scraped_text(brand_node.get_text(" ") if brand_node else "") or _infer_brand_from_title(title)

        price_meta = container.select_one('[itemprop="offers"] meta[itemprop="price"][content], meta[itemprop="price"][content]')
        price = _coerce_price(price_meta.get("content") if price_meta else None)

        container_text = clean_scraped_text(container.get_text(" "))
        class_text = " ".join(container.get("class", [])).casefold()
        availability = (
            "Indisponivel"
            if "not-available" in class_text or "esgotado" in container_text.casefold()
            else availability_text
        )
        combined_text = " ".join([title, brand or "", container_text])
        offer_payload = {
            "fornecedor": supplier_name,
            "descricao": title,
            "marca": brand or None,
            "unidade": _infer_commercial_unit(combined_text, fallback_unit),
            "quantidade": None,
            "valor_unitario": price,
            "valor_total": None,
            "preco_a_vista": None,
            "preco_a_prazo": None,
            "num_parcelas": default_installments,
            "frete": None,
            "disponibilidade": availability,
            "data_consulta": date.today().isoformat(),
            "link_produto": link_produto,
        }
        offers_by_url.setdefault(link_produto, offer_payload)

    offers = sorted(
        offers_by_url.values(),
        key=lambda offer: text_match_score(query, offer.get("descricao", ""), offer.get("unidade", "")),
        reverse=True,
    )
    _scraper_log(
        "schema_org_product_offers_extracted",
        {
            "supplier": supplier_name,
            "query": query,
            "offers": len(offers),
            "sample_titles": [offer.get("descricao") for offer in offers[:3]],
        },
    )
    return offers[:limit]


def parse_brazilian_price(price_text: str | None) -> float | None:
    """Parse a Brazilian currency string into float."""
    if not price_text:
        return None
    price_match = re.search(r"\d[\d\.]*,\d{2}", price_text)
    if not price_match:
        return None
    normalized = price_match.group(0).replace(".", "").replace(",", ".")
    try:
        return float(normalized)
    except ValueError:
        return None


def _same_site_url(url: str, base_url: str) -> bool:
    """Return whether url belongs to the same storefront host."""
    from urllib.parse import urlparse as _urlparse

    url_host = _urlparse(url).netloc.replace("www.", "")
    base_host = _urlparse(base_url).netloc.replace("www.", "")
    return bool(url_host and base_host and url_host == base_host)


def _infer_brand_from_title(title: str | None) -> str | None:
    """Infer brand from common title suffixes, e.g. Produto - CORAL."""
    if not title or " - " not in title:
        return None
    candidate = title.rsplit(" - ", 1)[-1].strip()
    if 2 <= len(candidate) <= 30:
        return candidate
    return None


def _candidate_container_text(anchor) -> str:
    """Climb a few levels from an anchor and return the nearest text containing a price."""
    node = anchor
    fallback_text = clean_scraped_text(anchor.get_text(" "))
    for _level in range(5):
        if node is None:
            break
        text = clean_scraped_text(node.get_text(" "))
        if "R$" in text:
            return text
        node = getattr(node, "parent", None)
    return fallback_text


def _title_from_candidate_text(anchor_text: str, container_text: str) -> str:
    """Extract product title from anchor/container text."""
    source_text = anchor_text if anchor_text and "R$" not in anchor_text else container_text
    title = re.split(r"\s+R\$\s*", source_text, maxsplit=1)[0]
    title = re.sub(r"\s+Por:\s*$", "", title, flags=re.IGNORECASE)
    return clean_scraped_text(title)


def scrape_storefront_search_offers(
    search_url: str,
    base_url: str,
    supplier_name: str,
    fallback_unit: str = "",
    availability_text: str = "Produto encontrado na busca do fornecedor.",
    default_installments: int | None = None,
    limit: int = 5,
) -> list[dict]:
    """Generic HTML storefront search scraper that returns OfertaFornecedor-shaped dicts."""
    _scraper_log(
        "storefront_html_search_start",
        {"supplier": supplier_name, "url": search_url, "limit": limit},
    )
    html = fetch_site_html(search_url)
    soup = parse_html(html)
    offers_by_url: dict[str, dict] = {}
    anchors_seen = 0
    price_candidates = 0

    for anchor in soup.find_all("a", href=True):
        anchors_seen += 1
        absolute_url = urljoin(base_url, anchor.get("href"))
        if not _same_site_url(absolute_url, base_url):
            continue
        if "/loja/busca.php" in absolute_url or "javascript:" in absolute_url:
            continue

        anchor_text = clean_scraped_text(anchor.get_text(" "))
        container_text = _candidate_container_text(anchor)
        price = parse_brazilian_price(container_text)
        if price is None:
            continue
        price_candidates += 1

        title = _title_from_candidate_text(anchor_text, container_text)
        if len(title) < 8:
            continue
        normalized_title = normalize_search_text(title)
        if normalized_title in {"comprar", "adicionar ao carrinho", "produto"}:
            continue
        if absolute_url in offers_by_url:
            continue

        combined_text = " ".join([title, container_text])
        offer_payload = {
            "fornecedor": supplier_name,
            "descricao": title,
            "marca": _infer_brand_from_title(title),
            "unidade": _infer_commercial_unit(combined_text, fallback_unit),
            "quantidade": None,
            "valor_unitario": price,
            "valor_total": None,
            "preco_a_vista": None,
            "preco_a_prazo": None,
            "num_parcelas": default_installments,
            "frete": None,
            "disponibilidade": availability_text,
            "data_consulta": date.today().isoformat(),
            "link_produto": absolute_url,
        }
        offers_by_url[absolute_url] = offer_payload
        _scraper_log(
            "storefront_html_offer_built",
            {
                "supplier": supplier_name,
                "descricao": offer_payload["descricao"],
                "valor_unitario": offer_payload["valor_unitario"],
                "unidade": offer_payload["unidade"],
                "link_produto": offer_payload["link_produto"],
            },
        )
        if len(offers_by_url) >= limit * 3:
            break

    offers = list(offers_by_url.values())
    _scraper_log(
        "storefront_html_search_done",
        {
            "supplier": supplier_name,
            "anchors_seen": anchors_seen,
            "price_candidates": price_candidates,
            "offers": len(offers),
            "sample_titles": [offer.get("descricao") for offer in offers[:3]],
        },
    )
    return offers[:limit]


GENERIC_STOREFRONT_LINK_TEXTS = {
    "comprar",
    "adicionar ao carrinho",
    "espiar",
    "ver detalhes",
    "produto",
    "todos os produtos",
    "departamentos",
    "categorias",
    "marcas",
    "preco",
    "voltar a pagina inicial",
}


def _is_generic_storefront_text(text: str | None) -> bool:
    """Return whether scraped text is storefront chrome instead of a product title."""
    normalized = normalize_search_text(text or "")
    if not normalized or normalized in GENERIC_STOREFRONT_LINK_TEXTS:
        return True
    generic_prefixes = (
        "product id",
        "product sku",
        "new in stock",
        "image",
        "input",
        "classificar por",
    )
    return normalized.startswith(generic_prefixes)


def _product_card_lines(anchor, max_levels: int = 5) -> tuple[list[str], bool]:
    """Return nearby product-card text lines and whether a product marker was found."""
    node = anchor
    fallback_lines = [clean_scraped_text(anchor.get_text(" "))]
    markers = ("product id", "product sku", "esgotado", "espiar", "comprar", "r$")
    for _level in range(max_levels):
        if node is None:
            break
        raw_lines = getattr(node, "get_text")("\n")
        lines = [clean_scraped_text(line) for line in raw_lines.splitlines()]
        lines = [line for line in lines if line]
        joined = " ".join(lines).lower()
        if any(marker in joined for marker in markers):
            return lines, True
        node = getattr(node, "parent", None)
    return [line for line in fallback_lines if line], False


def _best_title_from_product_lines(query: str, anchor_text: str, lines: list[str]) -> str:
    """Choose the most product-like title from a card's text lines."""
    candidates = []
    for raw_text in [anchor_text, *lines]:
        text = clean_scraped_text(raw_text)
        if len(text) < 8 or _is_generic_storefront_text(text):
            continue
        if re.fullmatch(r"R\$\s*\d[\d\.]*,\d{2}", text):
            continue
        title = _title_from_candidate_text(text, text)
        if len(title) < 8 or _is_generic_storefront_text(title):
            continue
        candidates.append(title)

    if not candidates:
        return ""
    return max(candidates, key=lambda title: (text_match_score(query, title), len(title)))


def scrape_storefront_product_card_offers(
    search_url: str,
    base_url: str,
    supplier_name: str,
    query: str,
    fallback_unit: str = "",
    availability_text: str = "Produto encontrado na busca do fornecedor.",
    default_installments: int | None = None,
    limit: int = 5,
) -> list[dict]:
    """Fallback scraper for storefronts that render product cards without visible prices."""
    _scraper_log(
        "storefront_product_card_search_start",
        {"supplier": supplier_name, "url": search_url, "query": query, "limit": limit},
    )
    html = fetch_site_html(search_url)
    soup = parse_html(html)
    offers_by_url: dict[str, dict] = {}
    anchors_seen = 0
    product_candidates = 0

    for anchor in soup.find_all("a", href=True):
        anchors_seen += 1
        absolute_url = urljoin(base_url, anchor.get("href"))
        if not _same_site_url(absolute_url, base_url):
            continue
        if "/loja/busca.php" in absolute_url or "javascript:" in absolute_url or "#" in absolute_url:
            continue

        anchor_text = clean_scraped_text(anchor.get_text(" "))
        lines, marker_found = _product_card_lines(anchor)
        if not marker_found:
            continue
        title = _best_title_from_product_lines(query=query, anchor_text=anchor_text, lines=lines)
        if not title or text_match_score(query, title) <= 0:
            continue
        if absolute_url in offers_by_url:
            continue

        product_candidates += 1
        combined_text = " ".join([title, *lines])
        price = parse_brazilian_price(combined_text)
        availability = "Esgotado" if "esgotado" in combined_text.lower() else availability_text
        offer_payload = {
            "fornecedor": supplier_name,
            "descricao": title,
            "marca": _infer_brand_from_title(title),
            "unidade": _infer_commercial_unit(combined_text, fallback_unit),
            "quantidade": None,
            "valor_unitario": price,
            "valor_total": None,
            "preco_a_vista": None,
            "preco_a_prazo": None,
            "num_parcelas": default_installments,
            "frete": None,
            "disponibilidade": availability,
            "data_consulta": date.today().isoformat(),
            "link_produto": absolute_url,
        }
        offers_by_url[absolute_url] = offer_payload
        _scraper_log(
            "storefront_product_card_offer_built",
            {
                "supplier": supplier_name,
                "descricao": offer_payload["descricao"],
                "valor_unitario": offer_payload["valor_unitario"],
                "unidade": offer_payload["unidade"],
                "link_produto": offer_payload["link_produto"],
            },
        )
        if len(offers_by_url) >= limit * 3:
            break

    offers = sorted(
        offers_by_url.values(),
        key=lambda offer: text_match_score(query, offer.get("descricao", ""), offer.get("unidade", "")),
        reverse=True,
    )
    _scraper_log(
        "storefront_product_card_search_done",
        {
            "supplier": supplier_name,
            "anchors_seen": anchors_seen,
            "product_candidates": product_candidates,
            "offers": len(offers),
            "sample_titles": [offer.get("descricao") for offer in offers[:3]],
        },
    )
    return offers[:limit]


def fetch_site_json(url: str, timeout: int = 20) -> Any:
    """Fetch JSON from a site endpoint."""
    _scraper_log("http_json_request", {"url": url, "timeout": timeout})
    response = requests.get(url, headers=DEFAULT_SCRAPING_HEADERS, timeout=timeout)
    _scraper_log(
        "http_json_response",
        {
            "url": url,
            "status_code": response.status_code,
            "content_type": response.headers.get("content-type"),
            "bytes": len(response.content or b""),
        },
    )
    response.raise_for_status()
    return response.json()


def search_pisolar_vtex_products(query: str, limit: int = 5) -> list[dict]:
    """Search Pisolar through VTEX public search endpoints."""
    _scraper_log("pisolar_vtex_search_start", {"query": query, "limit": limit})
    to_index = max(limit - 1, 0)
    encoded_query_param = quote_plus(query)
    encoded_path = quote(query, safe="")
    urls = [
        urljoin(PISOLAR_BASE_URL, f"api/catalog_system/pub/products/search?ft={encoded_query_param}&_from=0&_to={to_index}"),
        urljoin(PISOLAR_BASE_URL, f"api/catalog_system/pub/products/search/{encoded_path}?_from=0&_to={to_index}"),
    ]

    for index, url in enumerate(urls, start=1):
        _scraper_log("pisolar_vtex_endpoint_try", {"index": index, "url": url})
        try:
            products = fetch_site_json(url)
        except Exception as exc:
            _scraper_log("pisolar_vtex_endpoint_failed", {"url": url, "error_type": type(exc).__name__, "error": str(exc)})
            continue
        if isinstance(products, list):
            _scraper_log(
                "pisolar_vtex_endpoint_products",
                {
                    "url": url,
                    "count": len(products),
                    "sample_titles": [product.get("productName") for product in products[:3]],
                },
            )
        else:
            _scraper_log("pisolar_vtex_unexpected_payload", {"url": url, "payload_type": type(products).__name__})
        if isinstance(products, list) and products:
            _scraper_log("pisolar_vtex_search_success", {"query": query, "returned": min(len(products), limit)})
            return products[:limit]

    _scraper_log("pisolar_vtex_search_empty", {"query": query})
    return []


def _first_available_seller(item: dict) -> dict:
    sellers = item.get("sellers") or []
    if not sellers:
        _scraper_log("pisolar_item_without_sellers", {"item_name": item.get("nameComplete") or item.get("name")})
        return {}
    available = [
        seller
        for seller in sellers
        if (seller.get("commertialOffer") or {}).get("AvailableQuantity", 0) > 0
    ]
    selected_seller = available[0] if available else sellers[0]
    _scraper_log(
        "pisolar_seller_selected",
        {
            "item_name": item.get("nameComplete") or item.get("name"),
            "seller": selected_seller.get("sellerName"),
            "available_quantity": (selected_seller.get("commertialOffer") or {}).get("AvailableQuantity"),
        },
    )
    return selected_seller


def _best_vtex_item(product: dict) -> dict:
    items = product.get("items") or []
    if not items:
        _scraper_log("pisolar_product_without_items", {"product": product.get("productName")})
        return {}
    priced_items = []
    for item in items:
        seller = _first_available_seller(item)
        offer = seller.get("commertialOffer") or {}
        price = offer.get("Price")
        if price not in (None, 0):
            priced_items.append((price, item))
    if priced_items:
        selected_item = min(priced_items, key=lambda price_item: price_item[0])[1]
    else:
        selected_item = items[0]
    _scraper_log(
        "pisolar_item_selected",
        {
            "product": product.get("productName"),
            "items_count": len(items),
            "selected_item": selected_item.get("nameComplete") or selected_item.get("name"),
            "priced_items_count": len(priced_items),
        },
    )
    return selected_item


def _pisolar_product_to_offer(product: dict, fallback_unit: str = "") -> dict:
    item = _best_vtex_item(product)
    seller = _first_available_seller(item)
    offer = seller.get("commertialOffer") or {}
    title = product.get("productName") or item.get("nameComplete") or item.get("name") or ""
    description = clean_scraped_text(product.get("description") or product.get("metaTagDescription") or title)
    combined_text = " ".join(
        str(part)
        for part in [title, item.get("nameComplete"), item.get("complementName"), description]
        if part
    )
    price = offer.get("Price")
    if price in (None, 0):
        price = offer.get("spotPrice") or offer.get("ListPrice")
    installments = offer.get("Installments") or []
    installment_count = max((installment.get("NumberOfInstallments", 0) for installment in installments), default=None)
    available_quantity = offer.get("AvailableQuantity")

    offer_payload = {
        "fornecedor": seller.get("sellerName") or "Pisolar",
        "descricao": title,
        "marca": product.get("brand"),
        "unidade": _infer_commercial_unit(combined_text, fallback_unit),
        "quantidade": None,
        "valor_unitario": float(price) if price not in (None, "") else None,
        "valor_total": None,
        "preco_a_vista": None,
        "preco_a_prazo": None,
        "num_parcelas": installment_count or None,
        "frete": None,
        "disponibilidade": "Disponivel" if available_quantity and available_quantity > 0 else "Indisponivel",
        "data_consulta": date.today().isoformat(),
        "link_produto": product.get("link") or urljoin(PISOLAR_BASE_URL, f"{product.get('linkText', '')}/p"),
    }
    _scraper_log(
        "pisolar_offer_built",
        {
            "descricao": offer_payload["descricao"],
            "fornecedor": offer_payload["fornecedor"],
            "marca": offer_payload["marca"],
            "unidade": offer_payload["unidade"],
            "valor_unitario": offer_payload["valor_unitario"],
            "num_parcelas": offer_payload["num_parcelas"],
            "disponibilidade": offer_payload["disponibilidade"],
            "link_produto": offer_payload["link_produto"],
        },
    )
    return offer_payload


def scrape_pisolar_search_page(query: str, limit: int = 5) -> list[dict]:
    """Fallback scraper for Pisolar search pages when the VTEX API returns no products."""
    search_url = urljoin(PISOLAR_BASE_URL, f"{quote(query, safe='')}?map=ft")
    _scraper_log("pisolar_html_fallback_start", {"query": query, "url": search_url, "limit": limit})
    html = fetch_site_html(search_url)
    soup = parse_html(html)
    product_links = []
    seen_urls = set()
    anchors_seen = 0
    for anchor in soup.find_all("a", href=True):
        anchors_seen += 1
        absolute_url = urljoin(PISOLAR_BASE_URL, anchor.get("href"))
        if not re.search(r"/\d+/p(?:$|[?#])", absolute_url):
            continue
        if absolute_url in seen_urls:
            continue
        seen_urls.add(absolute_url)
        text = clean_scraped_text(anchor.get_text(" "))
        if text:
            product_links.append({"url": absolute_url, "title": text})
        if len(product_links) >= limit:
            break
    _scraper_log(
        "pisolar_html_candidates",
        {
            "query": query,
            "anchors_seen": anchors_seen,
            "candidate_count": len(product_links),
            "sample_titles": [product.get("title") for product in product_links[:3]],
        },
    )

    results = []
    for product in product_links:
        title = product["title"]
        offer_payload = {
                "fornecedor": "Pisolar",
                "descricao": title,
                "marca": None,
                "unidade": _infer_commercial_unit(title),
                "quantidade": None,
                "valor_unitario": None,
                "valor_total": None,
                "preco_a_vista": None,
                "preco_a_prazo": None,
                "num_parcelas": None,
                "frete": None,
                "disponibilidade": "Produto encontrado na busca da Pisolar.",
                "data_consulta": date.today().isoformat(),
                "link_produto": product["url"],
            }
        _scraper_log(
            "pisolar_html_offer_built",
            {
                "descricao": offer_payload["descricao"],
                "unidade": offer_payload["unidade"],
                "link_produto": offer_payload["link_produto"],
            },
        )
        results.append(offer_payload)

    _scraper_log("pisolar_html_fallback_done", {"query": query, "returned": len(results)})
    return results


@tool
def search_supplier_pisolar_tool(
    product_name: str,
    unit: str = "",
    quantity: float | None = None,
    profile: str = "Medio custo",
) -> list[dict]:
    """Search Pisolar through its site search and return supplier offers in the same shape as Serper.
    Pisolar é uma loja de materiais de construção com um portfólio amplo para obras residenciais
    oferecendo pisos e revestimentos cerâmicos, porcelanatos, argamassas e rejuntes, além de materiais hidráulicos 
    e elétricos, tintas, impermeabilizantes, ferramentas, iluminação, portas, janelas, telhas e itens de acabamento. 
    Considere que todos os produtos podem ser parcelados em 8x. 
    E o frete é grátis.
    """
    search_queries = build_supplier_search_queries(product_name=product_name, unit=unit, quantity=quantity)
    _scraper_log(
        "pisolar_tool_start",
        {
            "product_name": product_name,
            "unit": unit,
            "quantity": quantity,
            "profile": profile,
            "queries": search_queries,
            "base_url": PISOLAR_BASE_URL,
        },
    )

    products = []
    selected_query = search_queries[0] if search_queries else product_name
    for index, query in enumerate(search_queries, start=1):
        selected_query = query
        _scraper_log("pisolar_tool_query_attempt", {"index": index, "query": query})
        products = search_pisolar_vtex_products(query=query, limit=5)
        if products:
            _scraper_log("pisolar_tool_query_selected", {"query": query, "products": len(products)})
            break
    if products:
        results = [_pisolar_product_to_offer(product, fallback_unit=unit) for product in products[:5]]
    else:
        _scraper_log("pisolar_tool_api_empty_using_html_fallback", {"selected_query": selected_query})
        results = scrape_pisolar_search_page(query=selected_query, limit=5)

    _scraper_log(
        "pisolar_tool_done",
        {
            "product_name": product_name,
            "selected_query": selected_query,
            "returned": len(results),
            "sample_offers": [
                {
                    "descricao": result.get("descricao"),
                    "valor_unitario": result.get("valor_unitario"),
                    "unidade": result.get("unidade"),
                    "link_produto": result.get("link_produto"),
                }
                for result in results[:3]
            ],
        },
    )
    return results


COMERCIAL_ALIANCA_BASE_URL = "https://www.comercialalianca.com/"
COMERCIAL_ALIANCA_STORE_ID = "1387054"


def build_comercial_alianca_search_url(query: str, page: int = 1) -> str:
    """Build Comercial Alianca Tray search URL."""
    url = (
        f"{COMERCIAL_ALIANCA_BASE_URL.rstrip('/')}/loja/busca.php"
        f"?loja={COMERCIAL_ALIANCA_STORE_ID}"
        f"&palavra_busca={quote_plus(query)}"
    )
    if page > 1:
        url += f"&pg={page}"
    return url


def search_comercial_alianca_html_products(query: str, unit: str = "", limit: int = 5) -> list[dict]:
    """Search Comercial Alianca storefront HTML and return offers."""
    search_url = build_comercial_alianca_search_url(query=query)
    html = fetch_site_html(search_url)
    offers = scrape_tray_datalayer_offers(
        html=html,
        query=query,
        supplier_name="Comercial Alianca",
        base_url=COMERCIAL_ALIANCA_BASE_URL,
        fallback_unit=unit,
        default_installments=10,
        limit=limit,
    )
    if not offers:
        _scraper_log("comercial_alianca_datalayer_empty_using_html_fallback", {"query": query, "search_url": search_url})
        offers = scrape_storefront_search_offers(
            search_url=search_url,
            base_url=COMERCIAL_ALIANCA_BASE_URL,
            supplier_name="Comercial Alianca",
            fallback_unit=unit,
            availability_text="Produto encontrado na busca da Comercial Alianca.",
            default_installments=10,
            limit=limit,
        )
    scored_offers = sorted(
        offers,
        key=lambda offer: text_match_score(query, offer.get("descricao", ""), offer.get("unidade", "")),
        reverse=True,
    )
    _scraper_log(
        "comercial_alianca_search_ranked",
        {
            "query": query,
            "offers": len(scored_offers),
            "sample_titles": [offer.get("descricao") for offer in scored_offers[:3]],
        },
    )
    return scored_offers[:limit]


@tool
def search_supplier_comercial_alianca_tool(
    product_name: str,
    unit: str = "",
    quantity: float | None = None,
    profile: str = "Medio custo",
) -> list[dict]:
    """Search Comercial Alianca through its site search and return supplier offers in the same shape as Serper.
    Comercial Alianca e uma loja de materiais de construcao, pintura, eletrica, hidraulica,
    acabamentos, ferramentas e ferragens. A busca usa a barra de pesquisa do site, sem cidade.
    Considere que os produtos podem ser parcelados em ate 10x sem juros quando o site permitir.
    """
    search_queries = build_supplier_search_queries(product_name=product_name, unit=unit, quantity=quantity)
    _scraper_log(
        "comercial_alianca_tool_start",
        {
            "product_name": product_name,
            "unit": unit,
            "quantity": quantity,
            "profile": profile,
            "queries": search_queries,
            "base_url": COMERCIAL_ALIANCA_BASE_URL,
        },
    )

    results = []
    selected_query = search_queries[0] if search_queries else product_name
    for index, query in enumerate(search_queries, start=1):
        selected_query = query
        _scraper_log("comercial_alianca_tool_query_attempt", {"index": index, "query": query})
        try:
            results = search_comercial_alianca_html_products(query=query, unit=unit, limit=5)
        except Exception as exc:
            _scraper_log(
                "comercial_alianca_tool_query_failed",
                {"query": query, "error_type": type(exc).__name__, "error": str(exc)},
            )
            continue
        if results:
            _scraper_log("comercial_alianca_tool_query_selected", {"query": query, "offers": len(results)})
            break

    _scraper_log(
        "comercial_alianca_tool_done",
        {
            "product_name": product_name,
            "selected_query": selected_query,
            "returned": len(results),
            "sample_offers": [
                {
                    "descricao": result.get("descricao"),
                    "valor_unitario": result.get("valor_unitario"),
                    "unidade": result.get("unidade"),
                    "link_produto": result.get("link_produto"),
                }
                for result in results[:3]
            ],
        },
    )
    return results


CASA_ELETRICIDADE_BASE_URL = "https://www.casadaeletricidade.com.br/"


def build_casa_eletricidade_search_url(query: str, page: int = 1) -> str:
    """Build Casa da Eletricidade Tray search URL."""
    url = (
        f"{CASA_ELETRICIDADE_BASE_URL.rstrip('/')}/loja/busca.php"
        f"?palavra_busca={quote_plus(query)}"
    )
    if page > 1:
        url += f"&pg={page}"
    return url


def search_casa_eletricidade_html_products(query: str, unit: str = "", limit: int = 5) -> list[dict]:
    """Search Casa da Eletricidade storefront HTML and return offers."""
    search_url = build_casa_eletricidade_search_url(query=query)
    html = fetch_site_html(search_url)
    offers = scrape_schema_org_product_offers(
        html=html,
        query=query,
        supplier_name="Casa da Eletricidade",
        base_url=CASA_ELETRICIDADE_BASE_URL,
        fallback_unit=unit,
        availability_text="Produto encontrado na busca da Casa da Eletricidade.",
        default_installments=None,
        limit=limit,
    )
    if not offers:
        _scraper_log("casa_eletricidade_schema_empty_using_datalayer", {"query": query, "search_url": search_url})
        offers = scrape_tray_datalayer_offers(
            html=html,
            query=query,
            supplier_name="Casa da Eletricidade",
            base_url=CASA_ELETRICIDADE_BASE_URL,
            fallback_unit=unit,
            default_installments=None,
            limit=limit,
        )
    if not offers:
        _scraper_log("casa_eletricidade_datalayer_empty_using_html_fallback", {"query": query, "search_url": search_url})
        offers = scrape_storefront_search_offers(
            search_url=search_url,
            base_url=CASA_ELETRICIDADE_BASE_URL,
            supplier_name="Casa da Eletricidade",
            fallback_unit=unit,
            availability_text="Produto encontrado na busca da Casa da Eletricidade.",
            default_installments=None,
            limit=limit,
        )
    scored_offers = sorted(
        offers,
        key=lambda offer: text_match_score(query, offer.get("descricao", ""), offer.get("unidade", "")),
        reverse=True,
    )
    _scraper_log(
        "casa_eletricidade_search_ranked",
        {
            "query": query,
            "offers": len(scored_offers),
            "sample_titles": [offer.get("descricao") for offer in scored_offers[:3]],
        },
    )
    if not scored_offers:
        _scraper_log("casa_eletricidade_search_price_empty", {"query": query, "search_url": search_url})
        scored_offers = scrape_storefront_product_card_offers(
            search_url=search_url,
            base_url=CASA_ELETRICIDADE_BASE_URL,
            supplier_name="Casa da Eletricidade",
            query=query,
            fallback_unit=unit,
            availability_text="Produto encontrado na busca da Casa da Eletricidade; preco nao visivel no HTML.",
            default_installments=None,
            limit=limit,
        )
    return scored_offers[:limit]


@tool
def search_supplier_casa_eletricidade_tool(
    product_name: str,
    unit: str = "",
    quantity: float | None = None,
    profile: str = "Medio custo",
) -> list[dict]:
    """Search Casa da Eletricidade through its site search and return supplier offers in the same shape as Serper.
    Use esta tool para materiais eletricos ou similares: fios, cabos, disjuntores, quadros,
    caixas eletricas, interruptores, tomadas, iluminacao LED, lampadas, chuveiros eletricos,
    torneiras eletricas, sensores, transformadores, fita isolante, equipamentos de seguranca,
    ferramentas, instrumentos de medicao, hidraulica, irrigacao e jardinagem.
    Nao use para pintura, pisos, alvenaria, cimento, areia, portas, janelas ou acabamento geral,
    exceto quando o material for explicitamente eletrico ou de uma categoria similar acima.
    A busca usa a barra de pesquisa do site, sem cidade.
    """
    search_queries = build_supplier_search_queries(product_name=product_name, unit=unit, quantity=quantity)
    _scraper_log(
        "casa_eletricidade_tool_start",
        {
            "product_name": product_name,
            "unit": unit,
            "quantity": quantity,
            "profile": profile,
            "queries": search_queries,
            "base_url": CASA_ELETRICIDADE_BASE_URL,
        },
    )

    results = []
    selected_query = search_queries[0] if search_queries else product_name
    for index, query in enumerate(search_queries, start=1):
        selected_query = query
        _scraper_log("casa_eletricidade_tool_query_attempt", {"index": index, "query": query})
        try:
            results = search_casa_eletricidade_html_products(query=query, unit=unit, limit=5)
        except Exception as exc:
            _scraper_log(
                "casa_eletricidade_tool_query_failed",
                {"query": query, "error_type": type(exc).__name__, "error": str(exc)},
            )
            continue
        if results:
            _scraper_log("casa_eletricidade_tool_query_selected", {"query": query, "offers": len(results)})
            break

    _scraper_log(
        "casa_eletricidade_tool_done",
        {
            "product_name": product_name,
            "selected_query": selected_query,
            "returned": len(results),
            "sample_offers": [
                {
                    "descricao": result.get("descricao"),
                    "valor_unitario": result.get("valor_unitario"),
                    "unidade": result.get("unidade"),
                    "link_produto": result.get("link_produto"),
                }
                for result in results[:3]
            ],
        },
    )
    return results


@tool
def search_supplier_serp_tool(
    product_name: str,
    unit: str = "",
    quantity: float | None = None,
    profile: str = "Medio custo",
    city: str = "Aracaju - SE"
) -> list[dict]:
    """Search for suppliers of a product using Serper Shopping and return the first 5 offers.
       Prefer supplier-specific tools such as Pisolar, Comercial Alianca or Casa da Eletricidade before Serper.
       Use somente quando não encontrar em tools específicas de fornecedores, como a tool da Pisolar
    """
    #serper_api_key = os.environ.get("SERPER_API_KEY") or os.environ.get("serper_api_key")
    serper_api_key = "677cf310c24bb42c30aa8634c3e3e64854d48038"
    if not serper_api_key:
        raise EnvironmentError("Defina SERPER_API_KEY ou serper_api_key antes de usar a busca Serper.")

    query_parts = [product_name, compact_quantity_unit_for_query(quantity=quantity, unit=unit), profile, city]

    query = " ".join(str(part).strip() for part in query_parts if str(part).strip())
    print(f"[tool:search_supplier_serp_tool] query='{query}' gl='br' hl='pt-br' num=5")

    response = requests.post(
        "https://google.serper.dev/shopping",
        headers={
            "X-API-KEY": serper_api_key,
            "Content-Type": "application/json",
        },
        json={
            "q": query,
            "gl": "br",
            "hl": "pt-br",
            "num": 5,
        },
        timeout=20,
    )
    print(f"[tool:search_supplier_serp_tool] status_code={response.status_code}")
    response.raise_for_status()
    payload = response.json()
    shopping_results = payload.get("shopping") or payload.get("shopping_results") or []

    def parse_price(price_text: str | None) -> float | None:
        if not price_text:
            return None
        price_match = re.search(r"\d[\d\.]*,?\d*", price_text)
        if not price_match:
            return None
        normalized = price_match.group(0).replace(".", "").replace(",", ".")
        try:
            return float(normalized)
        except ValueError:
            return None

    results = []
    for item in shopping_results[:5]:
        price = parse_price(item.get("price"))
        commercial_unit = _infer_commercial_unit(item.get("title"), unit)
        results.append(
            {
                "fornecedor": item.get("source") or item.get("seller") or "",
                "descricao": item.get("title"),
                "marca": None,
                "unidade": commercial_unit,
                "quantidade": None,
                "valor_unitario": price,
                "valor_total": None,
                "preco_a_vista": None,
                "preco_a_prazo": None,
                "num_parcelas": None,
                "frete": None,
                "disponibilidade": item.get("delivery") or item.get("availability"),
                "data_consulta": date.today().isoformat(),
                "link_produto": item.get("link") or item.get("product_link"),
            }
        )

    return results


In [97]:
# ReAct reasoning agent: ListaMateriaisObra -> ListaMateriaisObra with supplier options
class SupplierReasoningState(TypedDict):
    messages: Annotated[list, add_messages]


SUPPLIER_REASONING_SYSTEM_PROMPT = """
You are the base Reasoning agent for supplier discovery in Obra Barata.
Your job is to control the available tools, search supplier offers for one material at a time,
compare the evidence, and return only a JSON object that can update MaterialObra.

Rules:
- Use tools before choosing suppliers whenever a search tool is available.
- Choose supplier-specific tools by product domain before using a generic web search.
- Use search_supplier_casa_eletricidade_tool, when available, only for electrical or adjacent materials: fios, cabos, disjuntores, quadros de distribuicao, caixas eletricas, interruptores, tomadas, iluminacao/LED, lampadas, chuveiros ou torneiras eletricas, sensores, transformadores, fita isolante, equipamentos de seguranca/comunicacao, ferramentas eletricas/manuais, instrumentos de medicao, hidraulica, irrigacao e jardinagem.
- Do not call search_supplier_casa_eletricidade_tool for paint/coatings, floors, masonry, cement, sand, doors/windows, or general finishing unless the material is explicitly electrical or in an adjacent category above.
- Prefer supplier-specific tools before Serper: Casa da Eletricidade for electrical/similar items; Pisolar and Comercial Alianca for broader construction/finishing items; Serper only when supplier-specific tools are unavailable or return no good offers.
- Use tools to make calculations, conversions, and percentage computations; do not calculate in your head, when possible.
- Do not invent suppliers, prices, links, freight, installments, or availability.
- Prefer offers that match product name, unit,and product profile.
- When you recieve a list of offers, choose the best based on location, name match, unit match, and price.
- Do not stop after the first supplier tool if another relevant supplier-specific search tool is available.
- For broad construction, finishing, painting, tools, hydraulic or similar materials, call both search_supplier_pisolar_tool and search_supplier_comercial_alianca_tool when both are available, then compare the offers.
- For electrical or adjacent materials, call search_supplier_casa_eletricidade_tool when available; if it does not provide enough good priced offers, also call other relevant supplier tools.
- The top-level fornecedor and prices must represent the best offer, but lista_fornecedores must be a short ranked list of alternatives for the same material.
- Keep distinct offers from the same store when brand, model, package size, unit, price or link differs, because those differences matter to the user.
- Include offers from different suppliers in lista_fornecedores whenever they are available and relevant, even if the cheapest offer is from only one supplier.
- When deciding purchase quantity, compare material.quantidade + material.medida with each offer.unidade.
- Treat offer.valor_unitario as the price of one commercial package described by offer.unidade/title, not necessarily the price of one material.medida.
- If material.quantidade=30 and material.medida='L', then an offer.unidade='30 L' means quantidade=1; offer.unidade='20 L' means quantidade=2. Always round up so the purchase covers the required amount.
- Use purchase_quantity_tool whenever material.quantidade, material.medida, offer.unidade and offer.valor_unitario are available.
- Recalculate valor_total, preco_a_vista and preco_a_prazo from package price * purchase quantidade when package size is known.
- If package size cannot be inferred from offer.unidade or title, keep quantidade=1 for that offer and explain the uncertainty in justificativa.
- Return JSON only, with this shape:
{
  "fornecedor": "best supplier name or empty string",
  "lista_fornecedores": [
    {
      "fornecedor": "supplier name",
      "descricao": "product description",
      "marca": "brand or null",
      "unidade": "commercial unit",
      "quantidade": 1,
      "valor_unitario": 0,
      "valor_total": 0,
      "preco_a_vista": 0,
      "preco_a_prazo": 0,
      "num_parcelas": 1,
      "frete": 0,
      "disponibilidade": "availability text",
      "data_consulta": "YYYY-MM-DD",
      "link_produto": "source URL"
    }
  ],
  "valor_unitario": 0,
  "valor_total": 0,
  "preco_a_vista": 0,
  "preco_a_prazo": 0,
  "num_parcelas": 1,
  "frete": 0,
  "justificativa": "short evidence-based explanation"
}
""".strip()


def _preview_for_log(value: Any, max_chars: int = 1200) -> str:
    """Return a compact printable preview for reasoning/tool logs."""
    if value is None:
        return ""
    if isinstance(value, str):
        text = value
    else:
        try:
            text = json.dumps(value, ensure_ascii=False, default=str)
        except TypeError:
            text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) > max_chars:
        return text[:max_chars] + "... [truncated]"
    return text


def _log_reasoning_event(title: str, payload: Any | None = None) -> None:
    print(f"\n[reasoning] {title}")
    if payload is not None:
        print(_preview_for_log(payload))


def _log_tool_calls(response) -> None:
    tool_calls = getattr(response, "tool_calls", None) or []
    if not tool_calls:
        _log_reasoning_event("LLM did not request tools")
        return

    _log_reasoning_event(f"LLM requested {len(tool_calls)} tool call(s)")
    for index, tool_call in enumerate(tool_calls, start=1):
        name = tool_call.get("name") if isinstance(tool_call, dict) else getattr(tool_call, "name", "unknown")
        args = tool_call.get("args") if isinstance(tool_call, dict) else getattr(tool_call, "args", None)
        print(f"[reasoning][tool_call:{index}] {name} args={_preview_for_log(args)}")


def build_supplier_reasoning_agent(
    reasoning_llm,
    supplier_search_tools: list | None = None,
    checkpointer: MemorySaver | None = None,
):
    """Build the base ReAct graph that controls supplier search tools."""
    tools = list(supplier_search_tools or [])
    llm_with_tools = reasoning_llm.bind_tools(tools)
    tool_node = ToolNode(tools)

    def reasoning_agent(state: SupplierReasoningState) -> dict:
        messages = state["messages"]
        last_message = messages[-1] if messages else None
        _log_reasoning_event(
            f"Calling LLM with {len(messages)} message(s); last={type(last_message).__name__}",
            _message_text(last_message) if last_message is not None else None,
        )
        response = llm_with_tools.invoke(state["messages"])
        _log_reasoning_event("LLM visible response", _message_text(response))
        _log_tool_calls(response)
        return {"messages": [response]}

    def tools_with_logging(state: SupplierReasoningState) -> dict:
        last_message = state["messages"][-1]
        _log_reasoning_event("Executing requested tool(s)")
        try:
            result = tool_node.invoke(state)
        except Exception as exc:
            _log_reasoning_event("Tool execution failed", {"error_type": type(exc).__name__, "error": str(exc)})
            raise
        tool_messages = result.get("messages", []) if isinstance(result, dict) else []
        for index, tool_message in enumerate(tool_messages, start=1):
            tool_name = getattr(tool_message, "name", None) or getattr(tool_message, "tool_call_id", "unknown")
            print(f"[reasoning][tool_result:{index}] {tool_name}: {_preview_for_log(_message_text(tool_message))}")
        return result

    graph = StateGraph(SupplierReasoningState)
    graph.add_node("reasoning_agent", reasoning_agent)
    graph.add_node("tools", tools_with_logging)
    graph.add_edge(START, "reasoning_agent")
    graph.add_conditional_edges("reasoning_agent", tools_condition)
    graph.add_edge("tools", "reasoning_agent")
    return graph.compile(checkpointer=checkpointer)


def _message_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def _extract_json_object(text: str) -> dict:
    fenced_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced_match:
        return json.loads(fenced_match.group(1))

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {"lista_fornecedores": [], "justificativa": text.strip()}
    return json.loads(text[start : end + 1])


def _material_prompt(
    area_name: str,
    material: MaterialObra,
    max_fornecedores_por_material: int = 3,
) -> str:
    payload = {
        "area": area_name,
        "material": material.model_dump(mode="json"),
        "data_consulta": date.today().isoformat(),
    }
    return (
        "Find supplier offers for this MaterialObra and return the JSON update only. "
        "Use material.quantidade + material.medida as the required amount, and use offer.unidade "
        "to decide how many commercial packages must be bought. "
        f"Return up to {max_fornecedores_por_material} offers in lista_fornecedores for this material. "
        "lista_fornecedores is the final alternatives list, not only the selected winner. "
        "Include different suppliers when available, and keep relevant same-store brand/package variants.\n"
        + json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    )


def _valid_offer_payloads(update_payload: dict) -> list[OfertaFornecedor]:
    raw_offers = update_payload.get("lista_fornecedores") or update_payload.get("ofertas") or []
    offers = []
    for raw_offer in raw_offers:
        if not isinstance(raw_offer, dict) or raw_offer.get("status") == "tool_not_implemented":
            continue
        try:
            offers.append(OfertaFornecedor.model_validate(raw_offer))
        except Exception as exc:
            print(f"Skipping invalid supplier offer: {exc}")
    return offers


def _best_offer(offers: list[OfertaFornecedor]) -> OfertaFornecedor | None:
    priced_offers = [offer for offer in offers if _offer_rank_value(offer) < float("inf")]
    if priced_offers:
        return min(priced_offers, key=_offer_rank_value)
    return offers[0] if offers else None


def _offer_rank_value(offer: OfertaFornecedor) -> float:
    """Return the best comparable price for ranking an offer."""
    for value in (offer.valor_total, offer.preco_a_vista, offer.preco_a_prazo, offer.valor_unitario):
        if value is not None:
            return float(value)
    return float("inf")


def _offer_identity(offer: OfertaFornecedor) -> tuple[str, str, str, str]:
    """Build a stable identity to avoid repeated offer cards."""
    return (
        normalize_search_text(offer.fornecedor),
        normalize_search_text(offer.link_produto or ""),
        normalize_search_text(offer.descricao or ""),
        normalize_search_text(offer.unidade or ""),
    )


def _raw_offer_identity(raw_offer: dict) -> tuple[str, str, str, str]:
    """Build a stable identity for raw dict offers before model validation."""
    return (
        normalize_search_text(raw_offer.get("fornecedor", "")),
        normalize_search_text(raw_offer.get("link_produto") or ""),
        normalize_search_text(raw_offer.get("descricao") or ""),
        normalize_search_text(raw_offer.get("unidade") or ""),
    )


def _enrich_offer_for_material(material: MaterialObra, offer: OfertaFornecedor) -> OfertaFornecedor:
    """Fill purchase quantity and totals when the commercial package unit is clear."""
    updates: dict[str, Any] = {}
    purchase_quantity = offer.quantidade

    if purchase_quantity is None and material.quantidade is not None and material.medida and offer.unidade:
        required_family = _parse_unit_family(material.medida)
        offer_size, offer_family = _parse_amount_unit(offer.unidade)
        if required_family and offer_family and required_family == offer_family and offer_size:
            purchase_quantity = math.ceil(material.quantidade / offer_size)
            updates["quantidade"] = float(purchase_quantity)

    if purchase_quantity is not None and offer.valor_unitario is not None:
        total_price = round(float(offer.valor_unitario) * float(purchase_quantity), 2)
        if offer.valor_total is None:
            updates["valor_total"] = total_price
        if offer.preco_a_vista is None:
            updates["preco_a_vista"] = total_price
        if offer.preco_a_prazo is None:
            updates["preco_a_prazo"] = total_price

    return offer.model_copy(update=updates) if updates else offer


def _select_offer_options(offers: list[OfertaFornecedor], limit: int | None) -> list[OfertaFornecedor]:
    """Rank offers, keep supplier diversity, then cap the alternatives list."""
    if not offers:
        return []
    if limit is None:
        limit = 3
    limit = max(int(limit), 0)
    if limit == 0:
        return []

    deduped: list[OfertaFornecedor] = []
    seen = set()
    for offer in offers:
        key = _offer_identity(offer)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(offer)

    ranked = sorted(
        deduped,
        key=lambda offer: (
            _offer_rank_value(offer),
            normalize_search_text(offer.fornecedor),
            normalize_search_text(offer.descricao or ""),
        ),
    )
    selected: list[OfertaFornecedor] = []
    selected_keys = set()
    selected_suppliers = set()

    def add_offer(offer: OfertaFornecedor) -> None:
        key = _offer_identity(offer)
        if key in selected_keys or len(selected) >= limit:
            return
        selected.append(offer)
        selected_keys.add(key)
        selected_suppliers.add(normalize_search_text(offer.fornecedor))

    add_offer(ranked[0])
    for offer in ranked[1:]:
        supplier_key = normalize_search_text(offer.fornecedor)
        if supplier_key not in selected_suppliers:
            add_offer(offer)
        if len(selected) >= limit:
            return selected
    for offer in ranked[1:]:
        add_offer(offer)
        if len(selected) >= limit:
            break
    return selected


def _supplier_tool_offer_payloads(messages: list) -> list[dict]:
    """Extract raw supplier offers from search tool messages."""
    offers: list[dict] = []
    for message in messages:
        tool_name = getattr(message, "name", None)
        if not tool_name or not str(tool_name).startswith("search_supplier_"):
            continue
        try:
            payload = json.loads(_message_text(message))
        except json.JSONDecodeError:
            continue
        raw_offers = payload if isinstance(payload, list) else payload.get("lista_fornecedores", []) if isinstance(payload, dict) else []
        for raw_offer in raw_offers:
            if isinstance(raw_offer, dict) and raw_offer.get("status") != "tool_not_implemented":
                offers.append(raw_offer)
    return offers


def _merge_supplier_offer_payloads(update_payload: dict, extra_offers: list[dict]) -> dict:
    """Merge LLM-selected offers with raw search-tool offers, preserving unique options."""
    if not extra_offers:
        return update_payload
    merged_payload = dict(update_payload)
    primary_offers = merged_payload.get("lista_fornecedores") or merged_payload.get("ofertas") or []
    if isinstance(primary_offers, dict):
        primary_offers = list(primary_offers.values())
    if not isinstance(primary_offers, list):
        primary_offers = []

    merged_offers = []
    seen = set()
    for raw_offer in [*primary_offers, *extra_offers]:
        if not isinstance(raw_offer, dict):
            continue
        key = _raw_offer_identity(raw_offer)
        if key in seen:
            continue
        seen.add(key)
        merged_offers.append(raw_offer)
    if merged_offers:
        merged_payload["lista_fornecedores"] = merged_offers
    return merged_payload


ELECTRICAL_SUPPLIER_TERMS = {
    "eletrica",
    "eletrico",
    "fio",
    "fios",
    "cabo",
    "cabos",
    "disjuntor",
    "disjuntores",
    "quadro",
    "tomada",
    "tomadas",
    "interruptor",
    "interruptores",
    "lampada",
    "lampadas",
    "led",
    "iluminacao",
    "chuveiro",
    "torneira eletrica",
    "sensor",
    "transformador",
    "fita isolante",
    "caixa eletrica",
    "eletroduto",
}


def _tool_name(tool) -> str:
    """Return a stable LangChain/plain-function tool name."""
    return getattr(tool, "name", None) or getattr(tool, "__name__", "")


def _is_electrical_supplier_material(area_name: str, material: MaterialObra) -> bool:
    """Return whether Casa da Eletricidade is a relevant supplier for the material."""
    haystack = normalize_search_text(f"{area_name} {material.nome} {material.descricao}")
    return any(term in haystack for term in ELECTRICAL_SUPPLIER_TERMS)


def _supplier_tool_relevant_for_material(tool_name: str, area_name: str, material: MaterialObra) -> bool:
    """Decide which supplier-specific tools should be present in the alternatives list."""
    if tool_name == "search_supplier_serp_tool":
        return False
    if tool_name == "search_supplier_casa_eletricidade_tool":
        return _is_electrical_supplier_material(area_name, material)
    return tool_name in {"search_supplier_pisolar_tool", "search_supplier_comercial_alianca_tool"}


def _payload_has_supplier_for_tool(update_payload: dict, tool_name: str) -> bool:
    """Check whether update_payload already contains offers from the supplier behind tool_name."""
    expected_terms = {
        "search_supplier_pisolar_tool": ("pisolar",),
        "search_supplier_comercial_alianca_tool": ("comercial alianca",),
        "search_supplier_casa_eletricidade_tool": ("casa da eletricidade",),
    }.get(tool_name, ())
    if not expected_terms:
        return False
    raw_offers = update_payload.get("lista_fornecedores") or update_payload.get("ofertas") or []
    if isinstance(raw_offers, dict):
        raw_offers = list(raw_offers.values())
    if not isinstance(raw_offers, list):
        return False
    suppliers = [normalize_search_text(raw_offer.get("fornecedor", "")) for raw_offer in raw_offers if isinstance(raw_offer, dict)]
    return any(any(term in supplier for term in expected_terms) for supplier in suppliers)


def _raw_offers_from_tool_result(result: Any) -> list[dict]:
    """Normalize a direct tool result into raw offer dicts."""
    if isinstance(result, str):
        try:
            result = json.loads(result)
        except json.JSONDecodeError:
            return []
    raw_offers = result if isinstance(result, list) else result.get("lista_fornecedores", []) if isinstance(result, dict) else []
    return [raw_offer for raw_offer in raw_offers if isinstance(raw_offer, dict)]


def _invoke_supplier_tool_for_material(tool, material: MaterialObra) -> list[dict]:
    """Call a supplier search tool directly for fallback/complementary alternatives."""
    profile = getattr(material.perfil_produto, "value", None) or material.perfil_produto or "Medio custo"
    args = {
        "product_name": material.nome,
        "unit": material.medida or "",
        "quantity": material.quantidade,
        "profile": profile,
    }
    if hasattr(tool, "invoke"):
        return _raw_offers_from_tool_result(tool.invoke(args))
    return _raw_offers_from_tool_result(tool(**args))


def _missing_relevant_supplier_tool_offers(
    supplier_search_tools: list | None,
    area_name: str,
    material: MaterialObra,
    update_payload: dict,
) -> list[dict]:
    """Search relevant supplier-specific tools that the LLM did not include."""
    extra_offers: list[dict] = []
    for tool in supplier_search_tools or []:
        tool_name = _tool_name(tool)
        if not _supplier_tool_relevant_for_material(tool_name, area_name, material):
            continue
        if _payload_has_supplier_for_tool(update_payload, tool_name):
            continue
        _log_reasoning_event(
            "Complementary supplier search",
            {"tool": tool_name, "material": material.nome, "area": area_name},
        )
        try:
            tool_offers = _invoke_supplier_tool_for_material(tool, material)
        except Exception as exc:
            _log_reasoning_event(
                "Complementary supplier search failed",
                {"tool": tool_name, "error_type": type(exc).__name__, "error": str(exc)},
            )
            continue
        _log_reasoning_event(
            "Complementary supplier search result",
            {"tool": tool_name, "offers": len(tool_offers)},
        )
        extra_offers.extend(tool_offers)
    return extra_offers


def _apply_supplier_update(
    material: MaterialObra,
    update_payload: dict,
    max_fornecedores_por_material: int | None = 3,
) -> MaterialObra:
    offers = [
        _enrich_offer_for_material(material, offer)
        for offer in _valid_offer_payloads(update_payload)
    ]
    offers = _select_offer_options(offers, max_fornecedores_por_material)
    best_offer = _best_offer(offers)
    material_updates: dict[str, Any] = {}

    if offers:
        material_updates["lista_fornecedores"] = offers

    if update_payload.get("justificativa") not in (None, ""):
        material_updates["justificativa"] = update_payload["justificativa"]

    if best_offer is None and bool(update_payload.get("fornecedor")):
        for field in (
            "fornecedor",
            "valor_unitario",
            "valor_total",
            "preco_a_vista",
            "preco_a_prazo",
            "num_parcelas",
            "frete",
        ):
            value = update_payload.get(field)
            if value not in (None, ""):
                material_updates[field] = value

    if best_offer is not None:
        best_values = {
            "fornecedor": best_offer.fornecedor,
            "valor_unitario": best_offer.valor_unitario,
            "valor_total": best_offer.valor_total,
            "preco_a_vista": best_offer.preco_a_vista,
            "preco_a_prazo": best_offer.preco_a_prazo,
            "num_parcelas": best_offer.num_parcelas,
            "frete": best_offer.frete,
        }
        for field, value in best_values.items():
            if value not in (None, ""):
                material_updates[field] = value

    return material.model_copy(update=material_updates)


def reason_about_material_suppliers(
    agent,
    area_name: str,
    material: MaterialObra,
    thread_id: str,
    max_fornecedores_por_material: int = 3,
) -> dict:
    _log_reasoning_event(
        f"Starting supplier reasoning thread={thread_id}",
        {
            "area": area_name,
            "material": material.model_dump(mode="json"),
        },
    )
    result = agent.invoke(
        {
            "messages": [
                SystemMessage(content=SUPPLIER_REASONING_SYSTEM_PROMPT),
                HumanMessage(
                    content=_material_prompt(
                        area_name=area_name,
                        material=material,
                        max_fornecedores_por_material=max_fornecedores_por_material,
                    )
                ),
            ]
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    update_payload = _extract_json_object(_message_text(result["messages"][-1]))
    tool_offers = _supplier_tool_offer_payloads(result.get("messages", []))
    if tool_offers:
        update_payload = _merge_supplier_offer_payloads(update_payload, tool_offers)
        _log_reasoning_event(
            "Merged supplier search tool offers",
            {"tool_offers": len(tool_offers), "merged_offers": len(update_payload.get("lista_fornecedores", []))},
        )
    _log_reasoning_event("Extracted supplier update JSON", update_payload)
    return update_payload


def preencher_fornecedores_com_reasoning_agent(
    lista_materiais: ListaMateriaisObra,
    reasoning_llm=None,
    supplier_search_tools: list | None = None,
    max_materials: int | None = None,
    max_fornecedores_por_material: int | None = None,
    max_materiais_processados: int | None = None,
) -> ListaMateriaisObra:
    """Receive ListaMateriaisObra, run the ReAct reasoning agent, and fill supplier fields.

    max_materials is kept as the public shortcut for how many offers to keep in
    each material.lista_fornecedores. Use max_materiais_processados only when
    you need to limit how many materials from ListaMateriaisObra are processed.
    """
    offer_limit = max_fornecedores_por_material if max_fornecedores_por_material is not None else max_materials
    if offer_limit is None:
        offer_limit = 3
    offer_limit = max(int(offer_limit), 0)
    material_processing_limit = max_materiais_processados

    agent = build_supplier_reasoning_agent(
        reasoning_llm=reasoning_llm or llm,
        supplier_search_tools=supplier_search_tools,
        checkpointer=MemorySaver(),
    )
    _log_reasoning_event(
        "Supplier fill limits",
        {
            "max_fornecedores_por_material": offer_limit,
            "max_materiais_processados": material_processing_limit,
        },
    )

    processed = 0
    updated_areas = []
    for area_index, area in enumerate(lista_materiais.areas):
        updated_materials = []
        for material_index, material in enumerate(area.materiais):
            if material_processing_limit is not None and processed >= material_processing_limit:
                updated_materials.append(material)
                continue

            thread_id = f"supplier-reasoning-{area_index}-{material_index}-{hashlib.md5(material.nome.encode()).hexdigest()}"
            print(
                f"\n[reasoning] Processing material {processed + 1}: "
                f"area='{area.area}' nome='{material.nome}' medida='{material.medida}' "
                f"quantidade={material.quantidade} perfil='{material.perfil_produto}'"
            )
            update_payload = reason_about_material_suppliers(
                agent=agent,
                area_name=area.area,
                material=material,
                thread_id=thread_id,
                max_fornecedores_por_material=offer_limit,
            )
            complementary_offers = _missing_relevant_supplier_tool_offers(
                supplier_search_tools=supplier_search_tools,
                area_name=area.area,
                material=material,
                update_payload=update_payload,
            )
            if complementary_offers:
                update_payload = _merge_supplier_offer_payloads(update_payload, complementary_offers)
                _log_reasoning_event(
                    "Merged complementary supplier offers",
                    {
                        "complementary_offers": len(complementary_offers),
                        "merged_offers": len(update_payload.get("lista_fornecedores", [])),
                    },
                )
            updated_material = _apply_supplier_update(
                material=material,
                update_payload=update_payload,
                max_fornecedores_por_material=offer_limit,
            )
            _log_reasoning_event(
                "Applied supplier update",
                updated_material.model_dump(mode="json"),
            )
            updated_materials.append(updated_material)
            processed += 1

        updated_areas.append(area.model_copy(update={"materiais": updated_materials}))

    return lista_materiais.model_copy(update={"areas": updated_areas})

In [98]:
lista_materiais_quantificada = ListaMateriaisObra(
    areas=[
        {
            "area": "Pintura",
            "materiais": [
                MaterialObra(nome="Tinta acrílica branca", quantidade=30, medida="litros"),
                MaterialObra(nome="Rolo de pintura", quantidade=2, medida="unidades"),            ],
        },
        {
            "area": "Elétrica",
            "materiais": [
                MaterialObra(nome="Eletroduto 2,5mm²", quantidade=100, medida="metros"),
                MaterialObra(nome="Tomada 10A", quantidade=20, medida="unidades"),
            ],
        }
    ]
)


In [99]:
# Example, after you have a ListaMateriaisObra instance:
lista_com_fornecedores = preencher_fornecedores_com_reasoning_agent(
    lista_materiais=lista_materiais_quantificada,
    supplier_search_tools=[search_supplier_casa_eletricidade_tool, search_supplier_pisolar_tool, search_supplier_comercial_alianca_tool, purchase_quantity_tool, calculator_tool, percentage_tool, percent_change_tool],
    max_materials=3,  # ate 3 ofertas em lista_fornecedores para cada material
)



[reasoning] Supplier fill limits
{"max_fornecedores_por_material": 3, "max_materiais_processados": null}

[reasoning] Processing material 1: area='Pintura' nome='Tinta acrílica branca' medida='litros' quantidade=30.0 perfil='None'

[reasoning] Starting supplier reasoning thread=supplier-reasoning-0-0-05a2d652706751c56b87add1e0e8d44c
{"area": "Pintura", "material": {"nome": "Tinta acrílica branca", "descricao": "", "quantidade": 30.0, "medida": "litros", "fornecedor": "", "lista_fornecedores": [], "valor_unitario": null, "valor_total": null, "preco_a_vista": null, "preco_a_prazo": null, "num_parcelas": null, "frete": null, "perfil_produto": null, "origem": null, "justificativa": null, "nivel_confianca": null, "referencias_ifc": []}}

[reasoning] Calling LLM with 2 message(s); last=HumanMessage
Find supplier offers for this MaterialObra and return the JSON update only. Use material.quantidade + material.medida as the required amount, and use offer.unidade to decide how many commercial p

In [100]:
# lista_com_fornecedores to json    
print(json.dumps(lista_com_fornecedores.model_dump(mode="json"), ensure_ascii=False, indent=2))

{
  "obra": null,
  "responsavel": null,
  "data": null,
  "moeda": "BRL",
  "observacoes": null,
  "areas": [
    {
      "area": "Pintura",
      "materiais": [
        {
          "nome": "Tinta acrílica branca",
          "descricao": "",
          "quantidade": 30.0,
          "medida": "litros",
          "fornecedor": "Grupo Pisolar",
          "lista_fornecedores": [
            {
              "fornecedor": "Grupo Pisolar",
              "descricao": "Tinta Acrílica Pintalar 15L Branco Neve - Iquine",
              "marca": "TINTAS IQUINE",
              "unidade": "15 L",
              "quantidade": 2.0,
              "valor_unitario": 94.89,
              "valor_total": 189.78,
              "preco_a_vista": 189.78,
              "preco_a_prazo": 189.78,
              "num_parcelas": 8,
              "frete": 0.0,
              "disponibilidade": "Disponivel",
              "data_consulta": "2026-08-13",
              "link_produto": "https://www.pisolar.com.br/62248/p"
    